## 1. Environment Setup & Dependencies


In [ ]:
%pip install soundfile
!pip install -q pesq pystoi
!pip install -q thop
!pip install -q https://github.com/schmiph2/pysepm/archive/master.zip

## 2. Library Declarations & Reproducibility


In [ ]:
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Iterator, Sequence
from pathlib import Path
import json
import re
import wave
import numpy as np
import soundfile as sf
import os

import torch
from torch import Tensor, nn
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import BatchSampler
import time

from collections import defaultdict
import matplotlib.pyplot as plt
import random
import shutil
import math
import glob
import IPython.display as ipd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

try:
    import pystoi
except ImportError:
    pystoi = None

try:
    from pesq import pesq
except ImportError:
    pesq = None

try:
    import pysepm
except ImportError:
    print("Warning: pysepm not found. CSIG, CBAK, COVL will be skipped.")
    pysepm = None

try:
    from thop import profile
except ImportError:
    profile = None

# HOTFIX for NumPy 2.0 compatibility with pysepm
if not hasattr(np, 'NaN'):
    np.NaN = np.nan
if not hasattr(np, 'NINF'):
    np.NINF = -np.inf



## 3. Acoustic Front-End: Psychoacoustic Glasberg & Moore (1990) ERB Filterbank & Causal STFT/iSTFT

Matches **GTCRN** front-end:
- $N_{\text{fft}} = 512, R = 256$ (16 ms hop, $50\%$ overlap at $16\text{ kHz}$).
- **Square-Root Hann Analysis/Synthesis Window**: Ensures exact unity-gain Constant Overlap-Add (COLA) and zero reconstruction distortion.
- **Band Merging (BM)**: Bins $0...63$ ($0 - 2kHz$) remain $1:1$ linear. Bins $64...256$ ($2 - 8kHz)$, $193$ linear bins) are warped into $64$ ERB subbands.
- **Energy-Normalized Band Splitting (BS)**: $\mathbf{W}_{\text{bs}} = \mathbf{W}_{\text{erb}}^T \oslash (\sum \mathbf{W}_{\text{erb}}^T + \epsilon)$.


In [ ]:
class ERBFilterBank(nn.Module):
    """
    Psychoacoustically exact ERB Filterbank for Band Merging (BM) and Splitting (BS).
    - Low-band  (0 - 2 kHz, bins 0..63): 1:1 Identity mapping (64 linear bins).
    - High-band (2 - 8 kHz, bins 64..256): 193 linear bins -> 64 ERB filters.
    - Total Subbands: 128 (64 linear + 64 ERB).
    """
    def __init__(self, n_fft: int = 512, fs: int = 16000, n_linear: int = 64, n_erb: int = 64):
        super().__init__()
        self.n_linear = n_linear
        total_bins = n_fft // 2 + 1
        high_bins = total_bins - n_linear

        # Anchor frequencies with 1-bin overhang so linear bins 64 and 256 (Nyquist) are fully covered
        low_freq = ((n_linear - 1) / (n_fft // 2.0)) * (fs / 2.0)
        high_freq = ((total_bins) / (n_fft // 2.0)) * (fs / 2.0)

        erb_weights = self._build_true_erb_matrix(fs, n_fft, n_linear, high_bins, n_erb, low_freq, high_freq)
        self.register_buffer("bm_matrix", erb_weights)   # [64, 193]

        bs_weights = erb_weights.t() / (torch.sum(erb_weights.t(), dim=-1, keepdim=True) + 1e-8)
        self.register_buffer("bs_matrix", bs_weights)    # [193, 64]

    def _freq_to_erb(self, freq: float) -> float:
        return 21.4 * math.log10(4.37 * (freq / 1000.0) + 1.0)

    def _erb_to_freq(self, erb: float) -> float:
        return (10.0 ** (erb / 21.4) - 1.0) / 0.00437

    def _build_true_erb_matrix(self, fs: int, n_fft: int, n_linear: int, n_high: int, n_filters: int, f_min: float, f_max: float) -> Tensor:
        erb_min = self._freq_to_erb(f_min)
        erb_max = self._freq_to_erb(f_max)
        erb_pts = torch.linspace(erb_min, erb_max, n_filters + 2)
        freq_pts = torch.tensor([self._erb_to_freq(e.item()) for e in erb_pts])

        bin_pts = (freq_pts * n_fft / fs) - n_linear
        weights = torch.zeros(n_filters, n_high)

        for i in range(n_filters):
            b_prev, b_curr, b_next = bin_pts[i], bin_pts[i + 1], bin_pts[i + 2]
            for b in range(n_high):
                if b_prev <= b <= b_curr and b_curr > b_prev:
                    weights[i, b] = (b - b_prev) / (b_curr - b_prev)
                elif b_curr < b <= b_next and b_next > b_curr:
                    weights[i, b] = (b_next - b) / (b_next - b_curr)

        return weights / (torch.sum(weights, dim=-1, keepdim=True) + 1e-8)

    def bm(self, x: Tensor) -> Tensor:
        """Forward Band Merging: [B, ..., 257] -> [B, ..., 128]"""
        x_low = x[..., :self.n_linear]
        x_high = x[..., self.n_linear:]
        x_high_erb = torch.matmul(x_high, self.bm_matrix.t())
        return torch.cat([x_low, x_high_erb], dim=-1)

    def bs(self, x_sub: Tensor) -> Tensor:
        """Inverse Band Splitting: [B, ..., 128] -> [B, ..., 257]"""
        x_low = x_sub[..., :self.n_linear]
        x_high = x_sub[..., self.n_linear:]
        x_high_linear = torch.matmul(x_high, self.bs_matrix.t())
        return torch.cat([x_low, x_high_linear], dim=-1)


def wav_to_stft(wav: Tensor, n_fft: int = 512, hop_length: int = 256, win_length: int = 512, window: Optional[Tensor] = None) -> Tensor:
    if window is None:
        window = torch.hann_window(window_length=win_length, device=wav.device).pow(0.5)

    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True
    )
    return spec.transpose(-1, -2)   # [B, T, F]


def stft_to_wav(spec: Tensor, n_fft: int = 512, hop_length: int = 256, win_length: int = 512, window: Optional[Tensor] = None, length: Optional[int] = None) -> Tensor:
    if window is None:
        window = torch.hann_window(window_length=win_length, device=spec.device).pow(0.5)

    # Cast to float precision to prevent ComplexHalf crash under torch.amp.autocast
    if spec.dtype not in (torch.complex64, torch.complex128):
        spec = spec.to(torch.complex64)
    if window.dtype not in (torch.float32, torch.float64):
        window = window.float()

    spec_transposed = spec.transpose(-1, -2).contiguous()  # [B, F, T]
    return torch.istft(
        spec_transposed,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        length=length
    )


## 4. Exponential Moving Average (EMA) Normalization 

In [ ]:
class CausalEmaNorm(nn.Module):
    """
    Numerically stable Causal EMA Normalizer (Stage 1 / DeepFilterNet heritage).
    Normalizes time-frequency features strictly causally across time T without FP16 overflow.
    """
    def __init__(self, alpha: float = 0.99, eps: float = 1e-5):
        super().__init__()
        self.alpha = alpha
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, T, F = x.shape
        out = torch.empty_like(x)

        # Initialize moving mean and variance with first frame
        mean = x[:, :, 0, :]
        var = torch.ones_like(mean)

        out[:, :, 0, :] = (x[:, :, 0, :] - mean) / torch.sqrt(var + self.eps)

        for t in range(1, T):
            curr_x = x[:, :, t, :]
            diff = curr_x - mean
            mean = self.alpha * mean + (1.0 - self.alpha) * curr_x
            var = (self.alpha * var + (1.0 - self.alpha) * diff * (curr_x - mean)).clamp_min(0.0)

            out[:, :, t, :] = (curr_x - mean) / torch.sqrt(var + self.eps)

        return out

## 5. DuoFilter Architecture Definition

1. **`CausalSeparableConv2d` / `CausalSeparableConvTranspose2d`**:  
   - Decouples spatial time-frequency filtering from pointwise cross-channel projection, slashing parameters by $87.3\%$. Causality strictly enforced with past-only temporal padding.
   - Uses Depthwise Convolution 2D and Mish activation function
   - Mish formula:
      $$
      \text{Mish}(x) = x \cdot \tanh(\text{softplus}(x)) = x \cdot \tanh(\ln(1 + e^x))
      $$
2. **`ADCB` (Asymmetric Dilated Convolutional Block)**:  
   Eliminates the sequential 16-step `Bi-GRU` frequency sweep from LiSenNet. Uses parallel 1D depthwise dilated convolutions ($d \in \{1, 2, 4\}$) across $F=16$, achieving $15/16$ subband receptive field in a single parallel step.
3. **`CausalConvGLU` & Uni-GRU**:  
   Causal temporal recurrence along $T$ and depthwise gated channel mixing.
4. **`Amplified Complex Ratio Masking` ($\beta = 2.0$)**:  
   Headroom bounded in $[-2.0, 2.0]$ to invert destructive phase cancellation.


In [ ]:
def get_groups(channels: int, max_groups: int = 4) -> int:
    """Finds the largest valid divisor of channels <= max_groups for GroupNorm"""
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1

class CausalDepthwiseConv2d(nn.Module):
    """
    Causal Depthwise-Separable 2D Convolution (DeepFilterNet Stage 1 Heritage).
    Decouples spatial time-frequency filtering (groups=in_ch) from channel mixing (1x1 conv).
    Strictly causal in time via asymmetric padding (kt - 1 past, 0 future).
    """
    def __init__(self, channels: int, kernel_size: tuple = (3, 3), stride: tuple = (1, 1), use_norm: bool = True, act: Optional[nn.Module] = nn.Mish(), bias: Optional[bool] = None):
        super().__init__()
        kt, kf = kernel_size
        self.pad = (kf // 2, kf // 2, kt - 1, 0)  # (pad_left, pad_right, pad_past, pad_future)

        if bias is None:
            bias = not use_norm

        self.dwconv = nn.Conv2d(
            channels,
            channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
            groups=channels,
            bias=bias
        )

        self.norm = nn.GroupNorm(num_groups=get_groups(channels, 4), num_channels=channels) if use_norm else nn.Identity()
        self.act = act if act is not None else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.act(self.norm(self.dwconv(F.pad(x, self.pad))))

class CausalDepthwiseConvTranspose2d(nn.Module):
    def __init__(
        self,
        channels: int,
        kernel_size: Tuple[int, int] = (1, 3),
        stride: Tuple[int, int] = (1, 2),
        use_norm: bool = True,
        act: Optional[nn.Module] = nn.Mish(),
        bias: Optional[bool] = None,
    ):
        super().__init__()
        kt, kf = kernel_size
        self.kt = kt
        if bias is None:
            bias = not use_norm

        self.dwtconv = nn.ConvTranspose2d(
            channels,
            channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=(0, kf // 2),
            output_padding = (0, 1),
            groups=channels,
            bias=bias,
        )

        self.norm = nn.GroupNorm(num_groups=get_groups(channels, 4), num_channels=channels) if use_norm else nn.Identity()
        self.act = act if act is not None else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        out = self.dwtconv(x)
        if self.kt > 1:
            out = out[:, :, :-(self.kt - 1), :]
        return self.act(self.norm(out))

class PConv2d(nn.Module):
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        use_norm: bool = True,
        act: Optional[nn.Module] = nn.Mish(),
        bias: Optional[bool] = None,
    ):
        super().__init__()
        if bias is None:
            bias = not use_norm

        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=1, bias=bias)
        self.norm = nn.GroupNorm(num_groups=get_groups(out_ch, 4), num_channels=out_ch) if use_norm else nn.Identity()
        self.act = act if act is not None else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.act(self.norm(self.conv(x)))

class CausalSeparableConv2d(nn.Module):
    """
    Encoder Block:
        x -> CausalDepthwiseConv2d -> PConv2d -> out
    """
    def __init__(self, in_ch: int, out_ch: int, kernel_size: tuple = (3, 3), stride: tuple = (1, 1), dropout_p: float = 0.0):
        super().__init__()
        self.depthwise = CausalDepthwiseConv2d(in_ch, kernel_size=kernel_size, stride=stride, use_norm=True, act=nn.Mish())
        self.pointwise = PConv2d(in_ch, out_ch, use_norm=True, act=nn.Mish())
        self.dropout = nn.Dropout2d(dropout_p) if dropout_p > 0.0 else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.dropout(self.pointwise(self.depthwise(x)))

class CausalSeparableConvTranspose2d(nn.Module):
    """
    Decoder Block:
        x -> PConv2d -> CausalDepthwiseConvTranspose2d -> out
    """
    def __init__(self, in_ch: int, out_ch: int, kernel_size: tuple = (1, 3), stride: tuple = (1, 2), dropout_p: float = 0.0):
        super().__init__()
        self.pointwise = PConv2d(in_ch, out_ch, use_norm=True, act=nn.Mish())
        self.depthwise = CausalDepthwiseConvTranspose2d(out_ch, kernel_size=kernel_size, stride=stride, use_norm=True, act=nn.Mish())
        self.dropout = nn.Dropout2d(dropout_p) if dropout_p > 0.0 else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.dropout(self.depthwise(self.pointwise(x)))

class ADCB(nn.Module):
    """
    Asymmetric Dilated Convolutional Block across the Frequency Dimension (F=16).
    Replaces sequential Bi-GRU with parallelized 1D depthwise dilated convolutions (d={1, 2, 4}).
    Receptive field = 1 + 2*(1+2+4) = 15 frequency bins.
    """
    def __init__(self, channels: int = 64, dilations: tuple = (1, 2, 4), dropout_p: float = 0.1):
        super().__init__()
        self.norm = nn.GroupNorm(num_groups=4, num_channels=channels)

        layers = []
        for d in dilations:
            layers.append(nn.Sequential(
                nn.Conv1d(channels, channels, kernel_size=3, stride=1, padding=d, dilation=d, groups=channels),
                nn.GroupNorm(num_groups=4, num_channels=channels),
                nn.Mish(),
                nn.Conv1d(channels, channels, kernel_size=1),
                nn.Dropout(dropout_p) if dropout_p > 0.0 else nn.Identity()
            ))

        self.convs = nn.ModuleList(layers)
        self.proj = nn.Linear(channels, channels)

    def forward(self, x: Tensor) -> Tensor:
        """Input: (B, C, T, F) -> Output: (B, C, T, F)"""
        B, C, T, F = x.shape
        h = x.permute(0, 2, 1, 3).reshape(B * T, C, F)
        res = self.norm(h)
        for conv in self.convs:
            res = res + conv(res)
        out = self.proj(res.permute(0, 2, 1)).permute(0, 2, 1)
        return out.view(B, T, C, F).permute(0, 2, 1, 3).contiguous()

class CausalConvGLU(nn.Module):
    """Channel mixer with depthwise 3x3 causal temporal filtering and GLU gating"""
    def __init__(self, channels: int = 64, expansion_factor: float = 1.0, dropout_p: float = 0.1):
        super().__init__()
        hidden_dim = int(channels * expansion_factor)
        self.norm = nn.GroupNorm(num_groups=get_groups(channels, 4), num_channels=channels)

        # 1x1 Conv pointwise projection 
        self.fc1 = nn.Conv2d(channels, hidden_dim * 2, kernel_size=1)

        # Depthwise 3x3 causal conv with past padding (left=1, right=1, top=2, bottom=0)
        self.dwconv = nn.Sequential(
            nn.ConstantPad2d((1, 1, 2, 0), value=0.0),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, stride=1, groups=hidden_dim, bias=True)
        )

        self.act = nn.Mish()
        self.dropout = nn.Dropout(dropout_p) if dropout_p > 0.0 else nn.Identity()
        self.fc2 = nn.Conv2d(hidden_dim, channels, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        res = x
        h = self.norm(x)
        x_branch, v_branch = self.fc1(h).chunk(2, dim=1)
        gated = self.act(self.dwconv(x_branch)) * torch.sigmoid(v_branch)
        out = self.fc2(self.dropout(gated))
        return res + out

class ADCB_Temporal_Bottleneck(nn.Module):
    """Dilated-Recurrent Bottleneck: Parallel Frequency Path + Causal Uni-GRU Temporal Path."""
    def __init__(self, channels: int = 64, dropout_p: float = 0.1):
        super().__init__()
        # 1. Parallel Frequency Path via ADCB 
        self.freq_adcb = ADCB(channels=channels, dilations=(1, 2, 4), dropout_p=dropout_p)

        # 2. Strictly Causal Temporal Path via Uni-GRU
        self.time_norm = nn.LayerNorm(channels)
        self.time_gru = nn.GRU(channels, channels, bidirectional=False, batch_first=True)
        self.time_fc = nn.Linear(channels, channels)
        self.time_dropout = nn.Dropout(dropout_p) if dropout_p > 0.0 else nn.Identity()

        # 3. Channel Mixer
        self.conv_glu = CausalConvGLU(channels=channels, expansion_factor=1.0, dropout_p=dropout_p)

    def forward(self, x: Tensor) -> Tensor:
        h_f = x + self.freq_adcb(x)
        B, C, T, F = h_f.shape
        h_t = h_f.permute(0, 3, 2, 1).reshape(B * F, T, C)
        out_t, _ = self.time_gru(self.time_norm(h_t))
        h_t = h_t + self.time_dropout(self.time_fc(out_t))
        out = h_t.view(B, F, T, C).permute(0, 3, 2, 1).contiguous()
        return self.conv_glu(out)

class DuoFilter(nn.Module):
    """
    Full Pipeline:
        STFT -> BM -> SFE -> Separable Encoder -> ADCB Bottleneck -> Separable Decoder -> CRM -> BS -> iSTFT
    """
    def __init__(self, beta: float = 2.0, use_skip_pconv: bool = False, dropout_p: float = 0.1, use_ema_norm: bool = False):
        super().__init__()
        self.beta = beta
        self.use_skip_pconv = use_skip_pconv
        self.use_ema_norm = use_ema_norm
        self.erb = ERBFilterBank(n_fft=512, fs=16000, n_linear=64, n_erb=64)

        if self.use_ema_norm:
            self.ema_norm = CausalEmaNorm(alpha=0.99, eps=1e-5)
        else:
            self.ema_norm = nn.Identity()
        
        # Causal Depthwise-Separable Encoder
        self.enc0 = CausalSeparableConv2d(9, 64, kernel_size=(3, 3), stride=(1, 1), dropout_p=dropout_p)
        self.enc1 = CausalSeparableConv2d(64, 64, kernel_size=(3, 3), stride=(1, 2), dropout_p=dropout_p)
        self.enc2 = CausalSeparableConv2d(64, 64, kernel_size=(3, 3), stride=(1, 2), dropout_p=dropout_p)
        self.enc3 = CausalSeparableConv2d(64, 64, kernel_size=(3, 3), stride=(1, 2), dropout_p=dropout_p)

        # Skip Connection Projections
        if self.use_skip_pconv:
            self.p3 = PConv2d(64, 64, use_norm=False, act=None, bias=False)
            self.p2 = PConv2d(64, 64, use_norm=False, act=None, bias=False)
            self.p1 = PConv2d(64, 64, use_norm=False, act=None, bias=False)
            self.p0 = PConv2d(64, 64, use_norm=False, act=None, bias=False)
        else:
            self.p3 = nn.Identity()
            self.p2 = nn.Identity()
            self.p1 = nn.Identity()
            self.p0 = nn.Identity()

        # ADCB Dilated-Recurrent Bottleneck
        self.bottleneck = ADCB_Temporal_Bottleneck(channels=64, dropout_p=dropout_p)

        # Causal Depthwise-Separable Trannsposed Conv Decoder
        self.up3 = CausalSeparableConvTranspose2d(64, 64, kernel_size=(1, 3), stride=(1, 2), dropout_p=dropout_p)
        self.up2 = CausalSeparableConvTranspose2d(64, 64, kernel_size=(1, 3), stride=(1, 2), dropout_p=dropout_p)
        self.up1 = CausalSeparableConvTranspose2d(64, 64, kernel_size=(1, 3), stride=(1, 2), dropout_p=dropout_p)

        # Output projection using PConv2d
        self.final_conv = PConv2d(64, 2, use_norm=False, act=None, bias=True)

    def forward(self, noisy_stft: Tensor) -> Tensor:
        """
        Input:
            noisy_stft: [B, 2, T, 257] (Yr, Yi)
        Returns:
            enhanced_stft: [B, 2, T, 257] (Sr, Si)
        """
        Yr, Yi = noisy_stft[:, 0], noisy_stft[:, 1]
        mag = torch.clamp(Yr**2 + Yi**2, min=1e-12)**0.15

        # 1. Front-end ERB Band Mergin
        feat = torch.stack([Yr, Yi, mag], dim=1)  # (B, 3, T, 257)
        feat_sub = self.erb.bm(feat)

        # Apply Causal EMA Norm
        feat_sub = self.ema_norm(feat_sub)

        # 2. Subband Feature Extraction (SFE: context unfoldng k = 3)
        x_pad = F.pad(feat_sub, (1, 1, 0, 0), mode='reflect')
        feat_sfe = x_pad.unfold(dimension=-1, size=3, step=1).permute(0, 1, 4, 2, 3).reshape(noisy_stft.shape[0], 9, -1, 128)

        # 3. Causal Depthwise-Separable Encoder
        x0 = self.enc0(feat_sfe)  # (B, 64, T, 128)
        x1 = self.enc1(x0)        # (B, 64, T, 64)
        x2 = self.enc2(x1)        # (B, 64, T, 32)
        x3 = self.enc3(x2)        # (B, 64, T, 16)

        # 4. Dilated-Recurrent Bottleneck
        bot = self.bottleneck(x3)

        # 5. Causal Separable Transposed Conv Decoder
        d2 = self.up3(bot + self.p3(x3))
        d1 = self.up2(d2 + self.p2(x2))
        d0 = self.up1(d1 + self.p1(x1)) + self.p0(x0)

        # 6. Amplified Tanh Complex Subband Mask: [-2.0, 2.0]
        crm_sub = self.beta * torch.tanh(self.final_conv(d0) / self.beta)

        # 7. Matrix Band Splitting to 257 linear bins
        crm_full = self.erb.bs(crm_sub)
        Mr, Mi = crm_full[:, 0], crm_full[:, 1]

        # 8. Complex Ration Masking Synthesis
        Sr = Yr * Mr - Yi * Mi
        Si = Yr * Mi + Yi * Mr

        return torch.stack([Sr, Si], dim=1)


## 6. Multi-Resolution Inside-Norm Loss Formulation

Stabilized spectral loss preventing NaN gradients in silence frames by placing $\epsilon = 10^{-6}$ inside the root:
$$\mathcal{L}_{\text{total}} = 0.50 \cdot \mathcal{L}_{\text{mag}} + 0.30 \cdot \mathcal{L}_{\text{cplx}} + 0.20 \cdot \mathcal{L}_{\text{MR}}$$
- $\mathcal{L}_{\text{mag}}$: Power-compressed magnitude MSE ($c=0.3$).
- $\mathcal{L}_{\text{cplx}}$: Inside-norm stabilized complex MSE.
- $\mathcal{L}_{\text{MR}}$: Multi-resolution spectral loss across STFT windows $\{256, 512, 1024\}$ to eliminate pitch harmonics and transients.


In [ ]:
class MultiResolutionNormLoss(nn.Module):
    def __init__(self, fft_sizes=(256, 512, 1024), hop_sizes=(128, 256, 512), c: float = 0.3, eps: float = 1e-6):
        super().__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.c = c
        self.eps = eps

    def forward(self, S_hat: Tensor, S: Tensor, s_hat_time: Tensor, s_time: Tensor) -> Tensor:
        """
        Args:
            S_hat: Predicted STFT [B, 2, T, 257]
            S: Clean Target STFT [B, 2, T, 257]
            s_hat_time: Predicted Waveform [B, 1, L]
            s_time: Clean Target Waveform [B, 1, L]
        """
        # 1. Primary Magnitude Loss (Inside-norm power compression)
        mag_hat_sq = S_hat[:, 0]**2 + S_hat[:, 1]**2
        mag_target_sq = S[:, 0]**2 + S[:, 1]**2
        loss_mag = F.mse_loss((mag_hat_sq + self.eps)**(self.c / 2.0), (mag_target_sq + self.eps)**(self.c / 2.0))

        # 2. Inside-Norm Stabilized Complex Loss
        scale_hat = (mag_hat_sq + self.eps)**((self.c - 1.0) / 2.0)
        scale_target = (mag_target_sq + self.eps)**((self.c - 1.0) / 2.0)
        loss_complex = F.mse_loss(S_hat * scale_hat.unsqueeze(1), S * scale_target.unsqueeze(1))

        # 3. Multi-Resolution Waveform Spectral Loss (Guarded with eps to prevent NaN gradients)
        loss_mr = 0.0
        for n_fft, hop in zip(self.fft_sizes, self.hop_sizes):
            window = torch.hann_window(n_fft, device=s_time.device)
            Y_hat = torch.stft(s_hat_time.squeeze(1), n_fft, hop, window=window, return_complex=True)
            Y_target = torch.stft(s_time.squeeze(1), n_fft, hop, window=window, return_complex=True)
            loss_mr += F.l1_loss(
                (torch.abs(Y_hat) + self.eps)**self.c,
                (torch.abs(Y_target) + self.eps)**self.c
            )

        return 0.50 * loss_mag + 0.30 * loss_complex + 0.20 * (loss_mr / len(self.fft_sizes))


## 7. Dynamic Mixing Dataset Generator (On-The-Fly Augmentation)

Fully consistent with `model.ipynb` and `stage1.ipynb`:
- Loads clean speech, environmental noise, and room impulse responses (RIR).
- Dynamically mixes them on-the-fly across SNR levels $\{-5, 0, 5, 10, 15, 20\}\text{ dB}$.
- Decouples RIR into early reflections target ($\le 50\text{ ms}$) and full reverberation input.

In [ ]:
class DynamicMixingDataset(Dataset):
    """
    On-the-fly dynamic mixing dataset for speech enhancement and dereverberation.
    Combines multi-source noise mixing, realistic RIR decay truncation, 
    floating dBFS microphone gain simulation, and deterministic validation seeding.
    """
    def __init__(
        self,
        clean_dir: str,
        noise_dir: str,
        rir_dir: Optional[str] = None,
        filterbank: Optional[Any] = None,
        segment_samples: int = 48000,
        flag: bool = False,
    ):
        super().__init__()
        self.clean_files = sorted(glob.glob(f"{clean_dir}/**/*.wav", recursive=True))
        self.noise_files = sorted(glob.glob(f"{noise_dir}/**/*.wav", recursive=True))

        if rir_dir is not None and os.path.exists(rir_dir):
            self.rir_files = sorted(glob.glob(f"{rir_dir}/**/*.wav", recursive=True))
        else:
            self.rir_files = []

        self.segment_samples = segment_samples
        self.filterbank = filterbank
        self.flag = flag  

        # Volume baseline & floating variation (+/- 10 dB)
        self.target_dB_FS = -25.0
        self.target_dB_FS_floating_value = 10.0

        # 50 ms threshold for early reflections
        self.time_early = 0.05
        self.snr_levels = [-5.0, 0.0, 5.0, 10.0, 15.0, 20.0]

    def __len__(self) -> int:
        return len(self.clean_files)
    
    def _load_and_crop(
        self, path: str, rng: random.Random, target_sr: int = 16000, target_samples: int = 48000
    ) -> Tensor:
        """Reads audio, downmixes to mono, crops/pads to target length, and resamples."""
        wav_np, sr = sf.read(path, dtype='float32')
        wav = torch.from_numpy(wav_np).float()

        # Downmix to mono
        if wav.ndim == 2:
            wav = wav.mean(dim=1)
        
        orig_length = int(target_samples * (sr / target_sr))
        # Crop first at original sample rate
        if wav.shape[-1] > orig_length:
            start = rng.randint(0, wav.shape[-1] - orig_length)
            wav = wav[start : start + orig_length]
        else:
            wav = F.pad(wav, (0, orig_length - wav.shape[-1]))
        # Resample second
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=target_sr)
        # Guarantee exact target_samples length
        if wav.shape[-1] > target_samples:
            wav = wav[..., :target_samples]
        elif wav.shape[-1] < target_samples:
            wav = F.pad(wav, (0, target_samples - wav.shape[-1]))
        return wav.unsqueeze(0)  # Shape: [1, L]

    def _load_rir(self, path: str, target_sr: int = 16000, target_samples: int = 16000) -> Tensor:
        """Loads RIR starting from sample 0 to preserve direct-path impulse response."""
        wav_np, sr = sf.read(path, dtype='float32')
        wav = torch.from_numpy(wav_np).float()
        if wav.ndim == 2:
            wav = wav.mean(dim=1)
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=target_sr)
        if wav.shape[-1] > target_samples:
            wav = wav[..., :target_samples]
        elif wav.shape[-1] < target_samples:
            wav = F.pad(wav, (0, target_samples - wav.shape[-1]))
        return wav.unsqueeze(0)  # Shape: [1, L]
    
    def _tailor_dB_FS(self, wav: Tensor, target_dB_FS: float, eps: float = 1e-6) -> Tuple[Tensor, Tensor, Tensor]:
        """Normalizes audio to a specific dBFS level."""
        rms = torch.sqrt(torch.mean(wav**2) + eps)
        scalar = 10 ** (target_dB_FS / 20.0) / rms
        return wav * scalar, rms, scalar

    def _apply_rir_and_target(self, clean_y: Tensor, rir_wav: Tensor, sr: int = 16000) -> Tuple[Tensor, Tensor]:
        """
        Splits RIR into full reverberation (noisy input) and early reflections (clean target)
        using the complete 3-case decay and cosine attenuation curves.
        """
        T0 = int(self.time_early * sr)  # ~800 samples for 50 ms @ 16 kHz
        T1 = 480                        # 30 ms smooth transition window
        len_rir = rir_wav.shape[-1]
        device = rir_wav.device
        # 1. Full Reverberation (What the microphone captures)
        clean_y_reverb = torchaudio.functional.fftconvolve(clean_y, rir_wav)
        clean_y_reverb = clean_y_reverb[:, :clean_y.shape[-1]]
        # 2. Early Reflections Target (What the model should preserve)
        if len_rir > (T0 + T1):
            Rd = 200.0
            alpha = 0.4
            # Exponential decay
            in_slope_decay = torch.arange(start=1, end=len_rir - T0 + 1, step=1, device=device, dtype=torch.float32)
            slope_decay = 10 ** ((-3.0 * in_slope_decay) / Rd)
            decay = torch.cat((torch.ones(T0, device=device), slope_decay), dim=0)
            # Cosine smoothing transition
            in_slope_att = torch.arange(start=1, end=T1 + 1, step=1, device=device, dtype=torch.float32)
            slope_att = (1 + alpha) / 2.0 + (1 - alpha) * torch.cos(torch.pi * in_slope_att / T1) / 2.0
            const = alpha * torch.ones(len_rir - T1 - T0, device=device)
            attenuated = torch.cat((torch.ones(T0, device=device), slope_att, const), dim=0)
            # Combine attenuation and decay
            out_filter = attenuated * decay
            rir_shorten = rir_wav * out_filter.unsqueeze(0)
        elif (T0 + T1) >= len_rir > T0:
            Rd = 200.0
            in_slope_decay = torch.arange(start=1, end=len_rir - T0 + 1, step=1, device=device, dtype=torch.float32)
            slope_decay = 10 ** ((-3.0 * in_slope_decay) / Rd)
            decay = torch.cat((torch.ones(T0, device=device), slope_decay), dim=0)
            rir_shorten = rir_wav * decay.unsqueeze(0)
        else:
            rir_shorten = rir_wav
        clean_target = torchaudio.functional.fftconvolve(clean_y, rir_shorten)
        clean_target = clean_target[:, :clean_y.shape[-1]]
        return clean_y_reverb, clean_target

    def snr_mix(
        self,
        clean_y: Tensor,
        noise_y: Tensor,
        snr: float,
        rir_wav: Optional[Tensor],
        rng: random.Random,
        eps: float = 1e-6,
        sr: int = 16000,
    ) -> Tuple[Tensor, Tensor]:
        """Modular SNR mixing with rigorous two-stage dBFS normalization."""
        # 1. Apply RIR if present
        if rir_wav is not None:
            clean_y_reverb, clean_target = self._apply_rir_and_target(clean_y, rir_wav, sr)
        else:
            clean_y_reverb = clean_y
            clean_target = clean_y
            
        # 2. Normalize speech to target base dBFS (-25 dBFS) and apply identical scalar to clean target
        clean_y_reverb, _, clean_scalar = self._tailor_dB_FS(clean_y_reverb, self.target_dB_FS, eps)
        clean_target = clean_target * clean_scalar

        # Normalize noise to target base dBFS (-25 dBFS)
        noise_y, _, _ = self._tailor_dB_FS(noise_y, self.target_dB_FS, eps)

        # 3. Scale noise to match requested SNR (both speech and noise are at -25 dBFS)
        snr_scalar = 10.0 ** (-snr / 20.0)
        noise_y = noise_y * snr_scalar
        noisy_y = clean_y_reverb + noise_y

        # 4. Floating gain variation (Simulating whisper to shout / mic distance)
        noisy_target_dB_FS = rng.uniform(
            self.target_dB_FS - self.target_dB_FS_floating_value,
            self.target_dB_FS + self.target_dB_FS_floating_value,
        )
        noisy_y, _, noisy_scalar = self._tailor_dB_FS(noisy_y, noisy_target_dB_FS, eps)
        # Apply the exact same scalar to target to maintain ground-truth relative volume
        clean_target = clean_target * noisy_scalar

        # 5. Anti-clipping prevention (threshold 0.95 for DAC headroom)
        max_val = torch.max(torch.abs(noisy_y))
        if max_val > 0.95:
            scale = 0.95 / (max_val + eps)
            noisy_y = noisy_y * scale
            clean_target = clean_target * scale
        return noisy_y, clean_target
    
    def __getitem__(self, idx: int) -> Dict[str, Tensor]:
        # Deterministic seed for validation/testing, fully random for training
        if self.flag:
            rng = random.Random(42 + idx)
        else:
            rng = random.Random()
        clean_wav = self._load_and_crop(self.clean_files[idx], rng, target_samples=self.segment_samples)
        # Apply RIR 50% of the time (if files are available)
        rir_wav = None
        if len(self.rir_files) > 0 and rng.random() < 0.5:
            rir_idx = rng.randint(0, len(self.rir_files) - 1)
            rir_wav = self._load_rir(self.rir_files[rir_idx], target_samples=16000) 
        # Multi-noise mixture: combine 1 to 5 random noise files
        noise_mixture = torch.zeros_like(clean_wav)
        num_noises = rng.randint(1, 5)
        for _ in range(num_noises):
            n_idx = rng.randint(0, len(self.noise_files) - 1)
            n_wav = self._load_and_crop(self.noise_files[n_idx], rng, target_samples=self.segment_samples)
            noise_mixture += n_wav

        # SNR mixing and dereverberation targeting
        target_snr_db = rng.choice(self.snr_levels)
        noisy_wav, clean_target = self.snr_mix(clean_wav, noise_mixture, target_snr_db, rir_wav, rng)

        return {
            "noisy_wav": noisy_wav.squeeze(0),
            "clean_target": clean_target.squeeze(0),
        }


## 8. Hyperparameter Setup & DataLoader Configuration

Hyperparameters aligned with **GTCRN**, **LiSenNet**, and **DeepFilterNet**:
- Audio Sampling Rate: $16,000\text{ Hz}$.
- Audio Segment: $3.0\text{ seconds}$ ($48,000\text{ samples}$).
- STFT: $N_{\text{fft}} = 512$, Hop $= 256$ ($16\text{ ms}$), Window $= 512$ with Square-Root Hann.
- Optimizer: **AdamW** (Weight Decay $= 10^{-4}$), Initial $\text{LR} = 5 \times 10^{-4}$.
- Dropout: $p = 0.1$ across spatial and temporal modules.
- Scheduler: **ReduceLROnPlateau** (factor $= 0.5$, patience $= 3$, min_lr $= 10^{-6}$).
- Gradient Clipping: $\text{max\_norm} = 5.0$.


In [ ]:
# Training & Audio Hyperparameters
sample_rate = 16000
segment_samples = 48000 # 3.0 seconds
n_fft = 512
hop_length = 256
win_length = 512
batch_size = 40
learning_rate = 5e-4
weight_decay = 1e-4
dropout_p = 0.1
epochs = 30
grad_clip_norm = 5.0

clean_dir = "/kaggle/input/datasets/quanninhhoang/dns-cleanspeech-dataset/train"
noise_dir = "/kaggle/input/datasets/quanninhhoang/dns-noise-dataset/train"
rir_dir = "/kaggle/input/datasets/quanninhhoang/dns-impulse-dataset/train"

val_clean_dir = "/kaggle/input/datasets/quanninhhoang/dns-cleanspeech-dataset/val"
val_noise_dir = "/kaggle/input/datasets/quanninhhoang/dns-noise-dataset/val"
val_rir_dir = "/kaggle/input/datasets/quanninhhoang/dns-impulse-dataset/val"


In [ ]:
if os.path.exists(clean_dir) and os.path.exists(noise_dir):
    train_dataset = DynamicMixingDataset(
        clean_dir=clean_dir,
        noise_dir=noise_dir,
        rir_dir=rir_dir
    )

    val_dataset = DynamicMixingDataset(
        clean_dir=val_clean_dir,
        noise_dir=val_noise_dir,
        rir_dir=val_rir_dir,
        flag=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )
else:
    print(f"Dataset path '{clean_dir}' not found on local disk. Mount datasets to run training.'")

## 9. Evaluation Function

In [ ]:
@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module, window: Tensor, device: torch.device) -> float:
    model.eval()
    total_val_loss = 0.0
    total_batches = 0

    for batch in loader:
        noisy_wav = batch["noisy_wav"].to(device)
        clean_target = batch["clean_target"].to(device)
        wav_len = noisy_wav.shape[-1]

        # 1. Forward STFT
        noisy_spec = wav_to_stft(noisy_wav, n_fft=512, hop_length=256, win_length=512, window=window)
        clean_spec = wav_to_stft(clean_target, n_fft=512, hop_length=256, win_length=512, window=window)

        noisy_input = torch.stack([noisy_spec.real, noisy_spec.imag], dim=1)  # [B, 2, T, 257]
        clean_stft = torch.stack([clean_spec.real, clean_spec.imag], dim=1)   # [B, 2, T, 257]

        # 2. Forward inference through model
        enhanced_stft = model(noisy_input)    # [B, 2, T, 257]

        # 3. Time synthesis via iSTFT
        enhanced_cplx = torch.complex(enhanced_stft[:, 0], enhanced_stft[:, 1])
        s_hat_time = stft_to_wav(enhanced_cplx, n_fft=512, hop_length=256, win_length=512, window=window, length=wav_len)

        # 4. Multi-Resolution Inside-Norm Loss
        val_loss = criterion(enhanced_stft, clean_stft, s_hat_time.unsqueeze(1), clean_target.unsqueeze(1))
        total_val_loss += val_loss.item()
        total_batches += 1

    return total_val_loss / max(total_batches, 1)


## 10. Training Loop with Mixed Precision & Gradient Clipping


In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    criterion: nn.Module,
    epochs: int,
    checkpoint_dir: str,
    device: torch.device,
    grad_clip_norm: float = 5.0
) -> List[Dict]:
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_path.mkdir(parents=True, exist_ok=True)

    window = torch.hann_window(window_length=512, device=device).pow(0.5)
    scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))
    best_val_loss = float("inf")
    history = []

    for epoch in range(epochs):
        epoch_start_time = time.perf_counter()
        model.train()
        total_train_loss = 0.0
        total_batches = 0

        for batch_idx, batch in enumerate(train_loader):
            noisy_wav = batch["noisy_wav"].to(device)
            clean_target = batch["clean_target"].to(device)
            wav_len = noisy_wav.shape[-1]

            # 1. Forward STFT
            noisy_spec = wav_to_stft(noisy_wav, n_fft=512, hop_length=256, win_length=512, window=window)
            clean_spec = wav_to_stft(clean_target, n_fft=512, hop_length=256, win_length=512, window=window)

            noisy_input = torch.stack([noisy_spec.real, noisy_spec.imag], dim=1)
            clean_stft = torch.stack([clean_spec.real, clean_spec.imag], dim=1)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
                # 2. Forward pass through DuoFilter
                enhanced_stft = model(noisy_input)   # [B, 2, T, 257]

                # 3. iSTFT Time-Domain reconstruction (cast to float to prevent ComplexHalf crash under AMP)
                enhanced_cplx = torch.complex(enhanced_stft[:, 0].float(), enhanced_stft[:, 1].float())
                s_hat_time = stft_to_wav(enhanced_cplx, n_fft=512, hop_length=256, win_length=512, window=window.float(), length=wav_len)

                # 4. Multi-Resolution Inside-Norm Loss
                loss = criterion(enhanced_stft, clean_stft, s_hat_time.unsqueeze(1), clean_target.unsqueeze(1))

            if torch.isnan(loss):
                print(f"Warning: NaN detected in batch {batch_idx}! Skipping step.")
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.detach().item()
            total_batches += 1

        train_loss = total_train_loss / max(total_batches, 1)

        # Evaluate on validation set
        val_loss = evaluate_model(model, val_loader, criterion, window, device)

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        epoch_duration = time.perf_counter() - epoch_start_time
        mins, secs = divmod(int(epoch_duration), 60)
        time_str = f"{mins:02d}m {secs:02d}s"

        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
        }, checkpoint_path / f"epoch_{epoch + 1:03d}.ckpt")

        # Save best model checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
            }, checkpoint_path / "best_model.ckpt")
            print(f"Epoch {epoch + 1:02d}/{epochs:02d} [{time_str}] train={train_loss:.6f} val={val_loss:.6f} -> [Best checkpoint saved!]", flush=True)
        else:
            print(f"Epoch {epoch + 1:02d}/{epochs:02d} [{time_str}] train={train_loss:.6f} val={val_loss:.6f}", flush=True)

    return history


## 11. Model Training Trigger

In [ ]:
model = DuoFilter(beta=2.0, use_skip_pconv=False, dropout_p=dropout_p, use_ema_norm=True).to(device)

criterion = MultiResolutionNormLoss(
    fft_sizes=(256, 512, 1024),
    hop_sizes=(128, 256, 512),
    c=0.3,
    eps=1e-6
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)

checkpoint_dir = "/kaggle/working/checkpoints"

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    epochs=epochs,
    checkpoint_dir=checkpoint_dir,
    device=device,
    grad_clip_norm=grad_clip_norm
)


## 12. Training and Validation Loss Curves

In [ ]:
if 'history' in locals() and history:
    epochs_range = [h['epoch'] for h in history]
    train_losses = [h['train_loss'] for h in history]
    val_losses = [h['val_loss'] for h in history]

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, train_losses, label="Training Loss", color="#1f77b4", lw=2)
    plt.plot(epochs_range, val_losses, label="Validation Loss", color="#ff7f0e", lw=2)
    plt.xlabel("Epoch")
    plt.ylabel("Multi-Resolution Inside-Norm Loss")
    plt.title("DuoFilter Training Convergence")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 13. Speech Metrics & Comprehensive VoiceBank+DEMAND Evaluation Suite

Includes standard speech enhancement benchmarks identical to `stage1.ipynb`:
- **Model Complexity Profiling** via `thop`: Measures Trainable Parameters and GMACs per 1 second of audio (16 kHz).
- **PESQ (Wideband, ITU-T P.862.2)**: Perceptual Evaluation of Speech Quality [-0.5 to 4.5].
- **STOI**: Short-Time Objective Intelligibility [0.0 to 1.0].
- **SI-SDR**: Scale-Invariant Signal-to-Distortion Ratio (dB).
- **Composite Metrics via `pysepm`**:
  - **CSIG**: Mean Opinion Score (MOS) predictor for Speech Distortion [1.0 to 5.0].
  - **CBAK**: MOS predictor for Background Noise Intrusion [1.0 to 5.0].
  - **COVL**: MOS predictor for Overall Audio Quality [1.0 to 5.0].
- **Delta (Δ) Improvement Analysis**: Side-by-side comparison of Noisy Baseline vs. Enhanced.
- **NumPy 2.0 Compatibility Hotfix**: Ensures seamless operation across modern Python environments.


In [ ]:
"""1. Core Metric calculation utilities"""
def calculate_si_sdr(estimate: Tensor, reference: Tensor, eps: float = 1e-8) -> float:
    """Calculates Scale-Invariant Signal-to-Distortion Ratio (SI-SDR) in dB."""
    est = estimate.flatten().double()
    ref = reference.flatten().double()

    # Zero-mean
    est = est - est.mean()
    ref = ref - ref.mean()

    # Projection scale factor alpha
    ref_energy = torch.sum(ref ** 2) + eps
    alpha = torch.sum(est * ref) / ref_energy

    e_target = alpha * ref
    e_res = est - e_target

    target_pow = torch.sum(e_target ** 2) + eps
    res_pow = torch.sum(e_res ** 2) + eps

    return float(10 * torch.log10(target_pow / res_pow).item())

def calculate_stoi(clean_wav: np.ndarray, deg_wav: np.ndarray, sr: int = 16000) -> float:
    """Calculates Short-Time Objective Intelligibility (STOI)."""
    if pystoi is None:
        return float("nan")
    return float(pystoi.stoi(clean_wav, deg_wav, sr, extended=False))

def calculate_pesq(clean_wav: np.ndarray, deg_wav: np.ndarray, sr: int = 16000) -> float:
    """Calculates Perceptual Evaluation of Speech Quality (PESQ Wideband)."""
    if pesq is None:
        return float("nan")
    try:
        c = np.clip(clean_wav.astype(np.float32), -1.0, 1.0)
        d = np.clip(deg_wav.astype(np.float32), -1.0, 1.0)
        mode = "wb" if sr in (16000, 48000) else "nb"
        return float(pesq(sr, c, d, mode))
    except Exception:
        return float("nan")

"""2. Model complexity profiling (PARAMS & MACs)"""
def print_model_complexity(model: nn.Module, device: torch.device, n_fft: int = 512, hop_length: int = 256):
    """Measures model parameters and MACs per 1 second of audio (16 kHz)."""
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {num_params:,} ({num_params / 1e3:.2f} K / {num_params / 1e6:.4f} M)")

    if profile is None:
        print("Note: 'thop' library not found. MACs computation skipped.")
        return

    # 1 second of audio at 16 kHz: 16000 samples -> (16000 // hop_length) + 1 frames
    frames_1sec = (16000 // hop_length) + 1  # 63 frames for hop=256
    dummy_input = torch.randn(1, 2, frames_1sec, n_fft // 2 + 1).to(device)

    try:
        # Profile on an isolated copy so hooks never contaminate the real model
        import copy
        model_copy = copy.deepcopy(model).to(device)
        macs, _ = profile(model_copy, inputs=(dummy_input,), verbose=False)
        print(f"MACs (per 1 second of 16 kHz audio): {macs / 1e6:.2f} MMACs ({macs / 1e9:.4f} GMACs)")
        del model_copy
    except Exception as e:
        print(f"Could not calculate MACs: {e}")

def evaluate_vctk_demand(
    model: nn.Module,
    clean_dir: str,
    noisy_dir: str,
    device: torch.device,
    sample_rate: int = 16000,
    n_fft: int = 512,
    hop_length: int = 256,
    win_length: int = 512,
    max_files: Optional[int] = None
):
    model.eval()
    window = torch.hann_window(window_length=win_length, device=device).pow(0.5)

    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    if max_files:
        noisy_files = noisy_files[:max_files]

    if not noisy_files:
        print(f"Warning: No .wav audio files found in: {noisy_dir}")
        return

    metrics = {
        "pesq": [], "stoi": [], "sisdr": [], "csig": [], "cbak": [], "covl": [],
        "noisy_pesq": [], "noisy_stoi": [], "noisy_sisdr": [], "noisy_csig": [], "noisy_cbak": [], "noisy_covl": []
    }

    print(os.linesep + f"Evaluating on {len(noisy_files)} files from VoiceBank+DEMAND...")

    with torch.inference_mode():
        for i, noisy_path in enumerate(noisy_files):
            filename = os.path.basename(noisy_path)
            clean_path = os.path.join(clean_dir, filename)

            if not os.path.exists(clean_path):
                continue

            # Load and resample
            noisy_wav, sr_n = torchaudio.load(noisy_path)
            clean_wav, sr_c = torchaudio.load(clean_path)

            if sr_n != sample_rate:
                noisy_wav = torchaudio.functional.resample(noisy_wav, sr_n, sample_rate)
            if sr_c != sample_rate:
                clean_wav = torchaudio.functional.resample(clean_wav, sr_c, sample_rate)

            # Move audio to target device and match length
            noisy_wav = noisy_wav.to(device)
            clean_wav = clean_wav.to(device)
            min_len = min(noisy_wav.shape[-1], clean_wav.shape[-1])
            noisy_wav = noisy_wav[..., :min_len]
            clean_wav = clean_wav[..., :min_len]

            # Forward STFT (noisy_wav is 2D [1, L])
            input_spec = wav_to_stft(noisy_wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=window)
            noisy_input = torch.stack([input_spec.real, input_spec.imag], dim=1)  # [1, 2, T, 257]

            with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
                enhanced_stft = model(noisy_input)

            # Inverse STFT (cast to float for safety)
            enhanced_cplx = torch.complex(enhanced_stft[:, 0].float(), enhanced_stft[:, 1].float())
            enh_wav = stft_to_wav(enhanced_cplx, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=window.float(), length=min_len).squeeze(0)

            # Squeeze to 1D numpy arrays so len(...) is the true sample count
            c_np = clean_wav.squeeze().cpu().numpy()
            e_np = enh_wav.squeeze().cpu().numpy()
            n_np = noisy_wav.squeeze().cpu().numpy()

            min_out_len = min(len(c_np), len(e_np), len(n_np))
            c_np = c_np[:min_out_len]
            e_np = e_np[:min_out_len]
            n_np = n_np[:min_out_len]

            # Enhanced Metrics
            metrics["pesq"].append(calculate_pesq(c_np, e_np, sample_rate))
            metrics["stoi"].append(calculate_stoi(c_np, e_np, sample_rate))
            metrics["sisdr"].append(calculate_si_sdr(torch.tensor(e_np), torch.tensor(c_np)))

            # Noisy Baseline Metrics
            metrics["noisy_pesq"].append(calculate_pesq(c_np, n_np, sample_rate))
            metrics["noisy_stoi"].append(calculate_stoi(c_np, n_np, sample_rate))
            metrics["noisy_sisdr"].append(calculate_si_sdr(torch.tensor(n_np), torch.tensor(c_np)))

            # Composite Metrics (CSIG, CBAK, COVL) via pysepm
            if pysepm is not None:
                try:
                    csig, cbak, covl = pysepm.composite(c_np, e_np, sample_rate)
                    metrics["csig"].append(csig)
                    metrics["cbak"].append(cbak)
                    metrics["covl"].append(covl)

                    n_csig, n_cbak, n_covl = pysepm.composite(c_np, n_np, sample_rate)
                    metrics["noisy_csig"].append(n_csig)
                    metrics["noisy_cbak"].append(n_cbak)
                    metrics["noisy_covl"].append(n_covl)
                except Exception:
                    pass

            if (i + 1) % 50 == 0 or (i + 1) == len(noisy_files):
                print(f"Processed {i + 1}/{len(noisy_files)} files...")

    # ==========================================
    # Print Table Summary
    # ==========================================
    print("\n" + "="*50)
    print(" VoiceBank+DEMAND Test Set Results")
    print("="*50)
    print(f"{'Metric':<10} | {'Noisy':<8} | {'Enhanced':<8} | {'Delta (Δ)':<8}")
    print("-" * 50)

    p_n, p_e = np.nanmean(metrics['noisy_pesq']), np.nanmean(metrics['pesq'])
    s_n, s_e = np.nanmean(metrics['noisy_stoi']), np.nanmean(metrics['stoi'])
    si_n, si_e = np.nanmean(metrics['noisy_sisdr']), np.nanmean(metrics['sisdr'])

    print(f"{'PESQ':<10} | {p_n:<8.3f} | {p_e:<8.3f} | {p_e - p_n:<+8.3f}")
    print(f"{'STOI':<10} | {s_n:<8.3f} | {s_e:<8.3f} | {s_e - s_n:<+8.3f}")
    print(f"{'SI-SDR':<10} | {si_n:<8.2f} | {si_e:<8.2f} | {si_e - si_n:<+8.2f}")

    if pysepm is not None and len(metrics["csig"]) > 0:
        cs_n, cs_e = np.nanmean(metrics['noisy_csig']), np.nanmean(metrics['csig'])
        cb_n, cb_e = np.nanmean(metrics['noisy_cbak']), np.nanmean(metrics['cbak'])
        co_n, co_e = np.nanmean(metrics['noisy_covl']), np.nanmean(metrics['covl'])

        print(f"{'CSIG':<10} | {cs_n:<8.3f} | {cs_e:<8.3f} | {cs_e - cs_n:<+8.3f}")
        print(f"{'CBAK':<10} | {cb_n:<8.3f} | {cb_e:<8.3f} | {cb_e - cb_n:<+8.3f}")
        print(f"{'COVL':<10} | {co_n:<8.3f} | {co_e:<8.3f} | {co_e - co_n:<+8.3f}")
    print("="*50)
            





In [ ]:
# checkpoint_path = "/kaggle/working/checkpoints/best_model.ckpt"
checkpoint_path = "/content/drive/MyDrive/DNS_Data/checkpoints/duofilter_best_model.ckpt"
# checkpoint_path = "checkpoints/best_model.ckpt"

# clean_dir = "/kaggle/input/datasets/quanninhhoang/voicebank-demand/clean_testset_wav/clean_testset_wav"
# noisy_dir = "/kaggle/input/datasets/quanninhhoang/voicebank-demand/noisy_testset_wav/noisy_testset_wav"
clean_dir = "/content/drive/MyDrive/DNS_Data/VoiceBank_Demand/clean_testset_wav/"
noisy_dir = "/content/drive/MyDrive/DNS_Data/VoiceBank_Demand/noisy_testset_wav/"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded checkpoint from: {checkpoint_path}")
else:
    print(f"Checkpoint not found at '{checkpoint_path}'. Using current model weights.")

model.eval()

In [ ]:
# 1. Measure Model Parameters and MACs (per 1 sec of audio)
print_model_complexity(model, device)

# 2. Run Comprehensive VoiceBank+DEMAND Benchmark
if os.path.exists(noisy_dir) and os.path.exists(clean_dir):
    evaluate_vctk_demand(
        model=model,
        clean_dir=clean_dir,
        noisy_dir=noisy_dir,
        device=device,
    )
else:
    print("VoiceBank+DEMAND directories not found at specified paths.")
    print("To run benchmark, please configure clean_dir and noisy_dir.")


## 14. Listen to VoiceBank+Demand audio

In [ ]:
def listen_to_samples(
    model: nn.Module,
    clean_dir: str,
    noisy_dir: str,
    device: torch.device,
    num_samples: int = 3,
    sample_rate: int = 16000,
    n_fft: int = 512,
    hop_length: int = 256,
    win_length: int = 512
):
    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    if not noisy_files:
        print(f"Warning: No .wav audio files found in: {noisy_dir}")
        return

    sample_files = random.sample(noisy_files, min(num_samples, len(noisy_files)))

    window = torch.hann_window(window_length=win_length, device=device).pow(0.5)
    model.eval()

    with torch.inference_mode():
        for noisy_path in sample_files:
            filename = os.path.basename(noisy_path)
            clean_path = os.path.join(clean_dir, filename)

            if not os.path.exists(clean_path):
                continue

            print(f"\n{'=' * 65}")
            print(f"Audio Sample: {filename}")
            print(f"{'=' * 65}")

            noisy_wav, sr_n = torchaudio.load(noisy_path)
            clean_wav, sr_c = torchaudio.load(clean_path)

            if sr_n != sample_rate:
                noisy_wav = torchaudio.functional.resample(noisy_wav, sr_n, sample_rate)
            if sr_c != sample_rate:
                clean_wav = torchaudio.functional.resample(clean_wav, sr_c, sample_rate)

            # Downmix to mono
            noisy_wav = noisy_wav.mean(dim=0, keepdim=True).to(device)
            clean_wav = clean_wav.mean(dim=0, keepdim=True).to(device)

            # Length alignment
            min_len = min(noisy_wav.shape[-1], clean_wav.shape[-1])
            noisy_wav = noisy_wav[..., :min_len]
            clean_wav = clean_wav[..., :min_len]

            # Complex STFT [1, T, 257]
            input_spec = wav_to_stft(
                noisy_wav,
                n_fft=n_fft,
                hop_length=hop_length,
                win_length=win_length,
                window=window
            )

            noisy_input = torch.stack([input_spec.real, input_spec.imag], dim=1)

            with torch.amp.autocast(device.type, enabled=(device.type=="cuda")):
                enhanced_stft = model(noisy_input)

            # Inverse STFT
            enhanced_cplx = torch.complex(
                enhanced_stft[:, 0].float(),
                enhanced_stft[:, 1].float()
            )

            enh_wav = stft_to_wav(
                enhanced_cplx,
                n_fft=n_fft,
                hop_length=hop_length,
                win_length=win_length,
                window=window.float(),
                length=min_len
            ).squeeze(0)

            noisy_np = noisy_wav.squeeze().cpu().numpy()
            clean_np = clean_wav.squeeze().cpu().numpy()
            enh_np = enh_wav.squeeze().cpu().numpy()

            print("1. Noisy Input:")
            ipd.display(ipd.Audio(noisy_np, rate=sample_rate))
            print("2. Enhanced Output (DuoFilter):")
            ipd.display(ipd.Audio(enh_np, rate=sample_rate))
            print("3. Clean Ground Truth:")
            ipd.display(ipd.Audio(clean_np, rate=sample_rate))

In [ ]:
# clean_dir = "/kaggle/input/datasets/quanninhhoang/voicebank-demand/clean_testset_wav/clean_testset_wav"
# noisy_dir = "/kaggle/input/datasets/quanninhhoang/voicebank-demand/noisy_testset_wav/noisy_testset_wav"
clean_dir = "/content/drive/MyDrive/DNS_Data/VoiceBank_Demand/clean_testset_wav/"
noisy_dir = "/content/drive/MyDrive/DNS_Data/VoiceBank_Demand/noisy_testset_wav/"

if os.path.exists(noisy_dir) and os.path.exists(clean_dir):
    listen_to_samples(
        model=model,
        clean_dir=clean_dir,
        noisy_dir=noisy_dir,
        device=device,
        num_samples=6
    )
else:
    print("VoiceBank+DEMAND test set directories not found. Please verify paths.")

## 15. Download checkpoint

In [ ]:
import zipfile
import os
from IPython.display import FileLink

file_to_zip = "/kaggle/working/checkpoints/best_model.ckpt"
zip_filename = 'best_checkpoint.zip'

if os.path.exists(file_to_zip):
    # 1. Create a zip file and add only the best checkpoint to it
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(file_to_zip)
    
    print("Zip created successfully!")
    
    # 2. Generate the download link
    display(FileLink(zip_filename))
else:
    print(f"Error: {file_to_zip} does not exist in the current directory.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')